In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,RobustScaler
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping
import os
import re
import joblib
import itertools
from IPython.display import display

# --- GPU Überprüfung und Setup ---
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print(f"NVIDIA GPU erkannt: {len(physical_devices)} GPU(s) verfügbar.")
    try:
        for gpu in physical_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth aktiviert (verhindert Out-of-Memory Fehler).")
    except RuntimeError as e:
        print(e)
else:
    print("Keine GPU erkannt. Training wird auf der CPU ausgeführt.")

df = pd.read_csv('training_data_ready.csv')
#df = df.drop([21,43,92,159], axis=0) #[21,43,92,159] # [92,99,31,169,150,43] #[17,21,43,53,62,92,129,150,159,166,169] #[15,17,21,30,35,43,53,62,67,72,73,75,91,92,129,139,142,150,159,166,169]
target_cols = ['x', 'y', 'z','Gewicht'] 
x_cols = [col for col in df.columns if col not in target_cols + ['source_file','Objektname','Objektkategorie','OAufnahme','Schwierigkeit','Gewicht']]
    
X = df[x_cols]
y = df[target_cols]
    # 3. Train-Test-Split (80% Training, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


NVIDIA GPU erkannt: 1 GPU(s) verfügbar.
GPU Memory Growth aktiviert (verhindert Out-of-Memory Fehler).


In [ ]:
folder_path = "Modelle"

pattern = r"model_config_\d+_(?P<scaler>[^_]+)_layers_(?P<layers>[\d-]+)_l2_(?P<l2>[\d.]+)_drop_(?P<drop>[\d.]+)_lr_(?P<lr>[\d.]+)_valloss_(?P<valloss>[\d.]+).keras"

data = []

if os.path.exists(folder_path):
    for filename in os.listdir(folder_path):
        match = re.match(pattern, filename)
        if match:
            info = match.groupdict()
            
            model_path = os.path.join(folder_path, filename)
            scaler_filename = filename.replace("model_", "scaler_").replace(".keras", ".pkl")
            scaler_path = os.path.join(folder_path, scaler_filename)
            
            try:
                model = tf.keras.models.load_model(model_path)
                scaler = joblib.load(scaler_path)
                X_train_scaled = scaler.transform(X_train)
                evaluation_results = model.evaluate(X_train_scaled, y_train, verbose=0)
                trainloss = evaluation_results[0] if isinstance(evaluation_results, list) else evaluation_results
                data.append({
                    "Modelgröße": info["layers"],
                    "scaler": info["scaler"],
                    "l2": float(info["l2"]),
                    "dropout": float(info["drop"]),
                    "lr": float(info["lr"]),
                    "trainloss": float(trainloss)
                })
            except Exception as e:
                print(f"Fehler beim Verarbeiten von {filename}: {e}")
else:
    print(f"Der Ordner '{folder_path}' wurde nicht gefunden.")

df = pd.DataFrame(data)

if not df.empty:
    print(f"{len(df)} passende Modell-Dateien verarbeitet.\n")

    pivot_table = df.pivot_table(
        index=["scaler", "l2", "dropout", "lr"],
        columns="Modelgröße",
        values="trainloss",
        aggfunc=lambda x: ", ".join(map(lambda v: f"{v:.5f}", sorted(x)))
    )
    print("--- Generierte Übersicht (MultiIndex) ---")
    print(pivot_table)

    excel_filename = "modell_uebersicht_hierarchisch.xlsx"
    pivot_table.to_excel(excel_filename, sheet_name="Modellübersicht", merge_cells=True, index=True, header=True)
    print(f"\nDie Excel-Datei wurde erfolgreich als '{excel_filename}' gespeichert.")

else:
    print("Es wurden keine passenden Dateien im Ordner gefunden oder verarbeitet.")

307 passende Modell-Dateien verarbeitet.

--- Generierte Übersicht (MultiIndex) ---
Modelgröße                           128-64-32          128-64-32-16  \
scaler l2  dropout lr                                                  
maxabs 0.0 0.0     0.001  158.67633, 260.54407  148.19247, 159.61032   
                   0.005  164.11319, 250.40205  175.07744, 196.74182   
                   0.010  127.10505, 234.54216  145.95778, 168.63069   
           0.1     0.001  140.59460, 285.23340  283.58435, 288.64117   
                   0.005             185.62660  302.38306, 338.73361   
                   0.010             228.90427  392.75220, 408.19354   
       0.1 0.0     0.001  309.53592, 313.08990  285.70471, 296.61893   
                   0.005  232.73975, 302.79489  185.49358, 192.34189   
                   0.010  178.64691, 233.89433  207.85435, 294.06516   
           0.1     0.001  298.98050, 313.75552  306.77280, 330.25589   
                   0.005  299.41528, 343.95648  334.

In [ ]:
folder_path = "Modelle3"

pattern = r"model_(?P<variant>.+?)_(?P<scaler>[^_]+)_layers_(?P<layers>[\d-]+)_l2_(?P<l2>[\d.]+)_drop_(?P<drop>[\d.]+)_lr_(?P<lr>[\d.]+)_valloss_(?P<valloss>[\d.]+)"

data = []

if os.path.exists(folder_path):
    for filename in os.listdir(folder_path):
        match = re.search(pattern, filename)
        if match:
            info = match.groupdict()

            model_path = os.path.join(folder_path, filename)

            scaler_filename = filename.replace("model_", "scaler_")
            if scaler_filename.endswith(".keras"):
                scaler_filename = scaler_filename[:-6] + ".pkl"
            else:
                scaler_filename = scaler_filename + ".pkl"

            scaler_path = os.path.join(folder_path, scaler_filename)

            try:
                model = tf.keras.models.load_model(model_path)
                scaler = joblib.load(scaler_path)
                X_train_current = X_train.copy()
                if "noA1" in info["variant"]:
                    cols_to_drop = [col for col in X_train_current.columns if "A1" in col or "A_1" in col]
                    X_train_current = X_train_current.drop(columns=cols_to_drop)
                X_train_scaled = scaler.transform(X_train_current)

                evaluation_results = model.evaluate(X_train_scaled, y_train, verbose=0)
                trainloss = evaluation_results[0] if isinstance(evaluation_results, list) else evaluation_results

                modellname = f"robust_{info['layers']}_l2_{info['l2']}_drop_{info['drop']}_lr_{info['lr']}"

                data.append({
                    "Variante": info["variant"],
                    "Modellname": modellname,
                    "trainloss": float(trainloss)
                })
            except Exception as e:
                print(f"Fehler beim Verarbeiten von {filename}: {e}")
else:
    print(f"Der Ordner '{folder_path}' wurde nicht gefunden.")

df = pd.DataFrame(data)

if not df.empty:
    print(f"{len(df)} passende Modell-Dateien in '{folder_path}' verarbeitet.\n")

    pivot_table = df.pivot_table(
        index="Variante",
        columns="Modellname",
        values="trainloss",
        aggfunc=lambda x: ", ".join(map(lambda v: f"{v:.5f}", sorted(x)))
    )

    print("--- Generierte Übersicht Modelle2 ---")
    print(pivot_table)

    excel_filename = "modell_uebersicht_modelle2.xlsx"
    pivot_table.to_excel(excel_filename, sheet_name="Modellübersicht 2", merge_cells=True, index=True, header=True)
    print(f"\nDie Excel-Datei wurde erfolgreich als '{excel_filename}' gespeichert.")

else:
    print("Es wurden keine passenden Dateien im Ordner gefunden oder verarbeitet.")

30 passende Modell-Dateien in 'Modelle3' verarbeitet.

--- Generierte Übersicht Modelle2 ---
Modellname    robust_128-64-32-16_l2_0.0_drop_0.1_lr_0.01  \
Variante                                                    
NoDrop_all                                       70.02237   
NoDrop_noA1                                      63.20247   
Variant1_all                                    140.62341   
Variant1_noA1                                   157.40019   
Variant2_all                                    114.70725   
Variant2_noA1                                   137.40778   

Modellname    robust_128-64-32-16_l2_0.1_drop_0.1_lr_0.01  \
Variante                                                    
NoDrop_all                                      124.42090   
NoDrop_noA1                                     230.45003   
Variant1_all                                    186.98381   
Variant1_noA1                                   286.91299   
Variant2_all                                    146.

In [ ]:
folder_path = "Modelle2"
model_files = [f for f in os.listdir(folder_path) if f.endswith('.keras')]
print(f"Gefundene Modelle: {len(model_files)}")

predictions = {}
valid_models = []

print("Lade Modelle und berechne Basis-Vorhersagen...")
for filename in model_files:
    model_path = os.path.join(folder_path, filename)

    # Scaler finden
    scaler_filename = filename.replace("model_", "scaler_")
    scaler_filename = scaler_filename.replace(".keras", ".pkl")
    if not scaler_filename.endswith(".pkl"):
        scaler_filename += ".pkl"
    scaler_path = os.path.join(folder_path, scaler_filename)

    try:
        model = tf.keras.models.load_model(model_path)
        scaler = joblib.load(scaler_path)

        X_train_current = X_train.copy()
        if "noA1" in filename:
            cols_to_drop = [col for col in X_train_current.columns if "A1" in col or "A_1" in col]
            X_train_current = X_train_current.drop(columns=cols_to_drop)

        X_train_scaled = scaler.transform(X_train_current)
        pred = model.predict(X_train_scaled, verbose=0)
        
        predictions[filename] = pred
        valid_models.append(filename)
    except Exception as e:
        print(f"Fehler bei {filename}: {e}")

model_aliases = {filename: f"M{i+1}" for i, filename in enumerate(valid_models)}
print("\n--- Modell-Legende ---")
for filename, alias in model_aliases.items():
    print(f"{alias}: {filename}")

def calculate_loss(y_true, y_pred):
    return np.mean(np.square(y_true - y_pred))

results = []
ensemble_sizes = [3]#, 5, 7]
y_train_arr = y_train.values

print("\nBerechne Ensembles (Mean & Median)...")
for k in ensemble_sizes:
    if k > len(valid_models):
        print(f"Warnung: Kann keine {k}-er Kombinationen bilden, da nur {len(valid_models)} Modelle geladen wurden.")
        continue
    allkombos = itertools.combinations(valid_models, k)

    for i, combo in enumerate(allkombos):

        combo_preds = np.array([predictions[m] for m in combo]) # Shape: (k, samples, targets)

        mean_preds = np.mean(combo_preds, axis=0)
        median_preds = np.median(combo_preds, axis=0)
        mean_loss = calculate_loss(y_train_arr, mean_preds)
        median_loss = calculate_loss(y_train_arr, median_preds)

        combo_alias = " + ".join([model_aliases[m] for m in combo])

        results.append({
            "Ensemble_Größe": k,
            "Kombination": combo_alias,
            "Mean_Loss": mean_loss,
            "Median_Loss": median_loss
        })

if results:
    df_ensemble = pd.DataFrame(results)

    print("\n--- Ergebnisse der Ensemble-Auswertung ---")
    for k in df_ensemble['Ensemble_Größe'].unique():
        print(f"\nTop 5 Ensembles für Größe {k} (sortiert nach Mean_Loss):")
        df_k = df_ensemble[df_ensemble['Ensemble_Größe'] == k]
        df_k = df_k.sort_values(by="Mean_Loss")
        display(df_k.head(5))

    excel_filename = "ensemble_kombinationen_auswertung.xlsx"
    df_ensemble = df_ensemble.sort_values(by=["Ensemble_Größe", "Mean_Loss"])
    df_ensemble.to_excel(excel_filename, index=False)
    print(f"\nDie gesamte Auswertung aller Kombinationen wurde in '{excel_filename}' gespeichert.")
else:
    print("Es konnten keine Ensembles gebildet werden.")

Gefundene Modelle: 64
Lade Modelle und berechne Basis-Vorhersagen...

--- Modell-Legende ---
M1: model_NoDrop_all_minmax_layers_256-128-64_l2_0.2_drop_0.0_lr_0.01_valloss_133.5947.keras
M2: model_NoDrop_all_minmax_layers_256-128-64_l2_0.2_drop_0.0_lr_0.01_valloss_78.0956.keras
M3: model_NoDrop_all_robust_layers_128-64-32-16_l2_0.0_drop_0.1_lr_0.01_valloss_212.0328.keras
M4: model_NoDrop_all_robust_layers_128-64-32-16_l2_0.0_drop_0.1_lr_0.01_valloss_65.9828.keras
M5: model_NoDrop_all_robust_layers_128-64-32-16_l2_0.0_drop_0.1_lr_0.01_valloss_73.0396.keras
M6: model_NoDrop_all_robust_layers_128-64-32-16_l2_0.1_drop_0.1_lr_0.01_valloss_106.0891.keras
M7: model_NoDrop_all_robust_layers_128-64-32-16_l2_0.1_drop_0.1_lr_0.01_valloss_265.2122.keras
M8: model_NoDrop_all_robust_layers_128-64-32-16_l2_0.1_drop_0.1_lr_0.01_valloss_94.2897.keras
M9: model_NoDrop_all_robust_layers_128-64-32_l2_0.1_drop_0.1_lr_0.01_valloss_109.6117.keras
M10: model_NoDrop_all_robust_layers_128-64-32_l2_0.1_drop_0.1_l

,Ensemble_Größe,Kombination,Mean_Loss,Median_Loss
947478,5,M2 + M16 + M23 + M24 + M53,13.896279,10.893816
941883,5,M2 + M16 + M17 + M23 + M53,13.919791,10.911884
904821,5,M2 + M14 + M16 + M17 + M53,14.266201,11.321901
905122,5,M2 + M14 + M16 + M24 + M53,14.349188,11.511490
941923,5,M2 + M16 + M17 + M24 + M53,14.357476,11.029325


ValueError: This sheet is too large! Your sheet size is: 7624512, 4 Max sheet size is: 1048576, 16384